In [ ]:
# Cell 1: Imports, Device & Wandb Setup
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

import wandb

# --- Paths (change DATA_DIR when running on Kaggle) ---
DATA_DIR = Path("/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/")

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Reproducibility ---
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- Weights & Biases ---
# One API key, reused forever — store it once (see note below), not per run.
def setup_wandb() -> None:
    api_key = os.environ.get("WANDB_API_KEY")

    if api_key is None:
        try:
            from kaggle_secrets import UserSecretsClient
            api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
            os.environ["WANDB_API_KEY"] = api_key
        except Exception:
            pass

    if api_key:
        os.environ["WANDB_API_KEY"] = api_key
        wandb.login()  # reads from env — same behavior, fewer warnings
    else:
        # Local dev: uses ~/.netrc after first `wandb login`, or prompts once
        wandb.login()


setup_wandb()

In [ ]:
# Cell 2: Custom PyTorch Dataset (FERDataset)
class FERDataset(Dataset):
    """Facial Expression Recognition dataset from Kaggle CSV format.

    Parses space-separated pixel strings into (1, 48, 48) grayscale tensors
    normalized to [0, 1]. Returns (image, label) for training data.
  """

    IMAGE_SIZE = 48
    NUM_PIXELS = IMAGE_SIZE * IMAGE_SIZE  # 2304

    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
        self.has_labels = "emotion" in self.dataframe.columns

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        pixel_str = self.dataframe.loc[idx, "pixels"]
        pixels = np.fromstring(pixel_str, sep=" ", dtype=np.float32)

        if pixels.size != self.NUM_PIXELS:
            raise ValueError(
                f"Expected {self.NUM_PIXELS} pixels, got {pixels.size} at index {idx}"
            )

        image = pixels.reshape(self.IMAGE_SIZE, self.IMAGE_SIZE)
        image = image / 255.0
        image = torch.from_numpy(image).unsqueeze(0)  # (1, 48, 48)

        if self.transform is not None:
            image = self.transform(image)

        if self.has_labels:
            label = int(self.dataframe.loc[idx, "emotion"])
            return image, label

        return image

In [ ]:
# Cell 3: Data Splits & Loaders
TRAIN_CSV = DATA_DIR / "train.csv"
BATCH_SIZE = 64
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Load full training dataframe
df = pd.read_csv(TRAIN_CSV)
print(f"Loaded {len(df)} samples from {TRAIN_CSV}")
print(f"Emotion distribution:\n{df['emotion'].value_counts().sort_index()}")

# Full dataset (no augmentation yet — added in Model 3)
full_dataset = FERDataset(df)

# 70/15/15 train/validation/test split with fixed seed
total_size = len(full_dataset)
train_size = int(total_size * TRAIN_RATIO)
val_size = int(total_size * VAL_RATIO)
test_size = total_size - train_size - val_size
generator = torch.Generator().manual_seed(RANDOM_SEED)

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size], generator=generator
)
print(
    f"Train samples: {len(train_dataset)} | "
    f"Val samples: {len(val_dataset)} | "
    f"Test samples: {len(test_dataset)}"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

In [ ]:
# Cell 4: Model 1 — BaselineCNN (intentionally underfitting)
class BaselineCNN(nn.Module):
    """Minimal 2-layer CNN (1 conv + 1 linear) designed to underfit.

    Very few filters and a single pooling stage limit representational capacity,
    making it structurally prone to underfitting on this dataset.
    """

    NUM_CLASSES = 7

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),  # (B, 8, 48, 48)
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),       # (B, 8, 24, 24)
        )
        self.classifier = nn.Linear(8 * 24 * 24, self.NUM_CLASSES)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


# Quick shape check
_model_test = BaselineCNN()
_dummy = torch.randn(2, 1, 48, 48)
print(f"BaselineCNN output shape: {_model_test(_dummy).shape}")  # (2, 7)
del _model_test, _dummy

In [ ]:
# Cell 5: Sanity Check — overfit a single small batch
def sanity_check(
    model: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
    num_images: int = 2,
    epochs: int = 50,
    lr: float = 1e-2,
    loss_threshold: float = 0.05,
) -> bool:
    """Forward/backward sanity check: overfit 1–2 images to ~100% accuracy."""
    model = model.to(device)
    model.train()

    images, labels = next(iter(dataloader))
    images = images[:num_images].to(device)
    labels = labels[:num_images].to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    print(f"Sanity check: {num_images} image(s), {epochs} epochs, lr={lr}")

    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            preds = outputs.argmax(dim=1)
            accuracy = (preds == labels).float().mean().item()

        if epoch % 10 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | Loss: {loss.item():.6f} | Acc: {accuracy * 100:.1f}%")

    passed = loss.item() < loss_threshold and accuracy == 1.0

    if passed:
        print("Sanity Check Passed")
    else:
        print(
            f"Sanity Check FAILED — final loss={loss.item():.6f}, "
            f"accuracy={accuracy * 100:.1f}%"
        )

    return passed


# Run once during setup — disable after BaselineCNN sanity check passed
RUN_BASELINE_SANITY_CHECK = False

if RUN_BASELINE_SANITY_CHECK:
    sanity_check(BaselineCNN(), train_loader, device)
else:
    print("Skipping BaselineCNN sanity check (already passed).")

In [ ]:
# Cell 6: Reusable train_epoch / evaluate loops
def train_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
) -> tuple[float, float]:
    """Run one training epoch. Returns (avg_loss, accuracy)."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    """Evaluate model on a dataloader. Returns (avg_loss, accuracy)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += batch_size

    return running_loss / total, correct / total


def build_dataloaders(batch_size: int) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Rebuild loaders when batch size changes during hyperparameter search."""
    loader_kwargs = {
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available(),
    }
    train = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, **loader_kwargs)
    val = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
    test = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
    return train, val, test


class TransformedSubset(Dataset):
    """Apply transforms to an existing subset (e.g. augmentation on train only)."""

    def __init__(self, subset: Dataset, transform=None):
        self.subset = subset
        self.transform = transform

    def __len__(self) -> int:
        return len(self.subset)

    def __getitem__(self, idx: int):
        image, label = self.subset[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


def build_augmented_dataloaders(
    batch_size: int,
    train_transform: transforms.Compose,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Train loader with augmentation; val/test unchanged."""
    loader_kwargs = {
        "num_workers": 0,
        "pin_memory": torch.cuda.is_available(),
    }
    train_aug = TransformedSubset(train_dataset, train_transform)
    train = DataLoader(train_aug, batch_size=batch_size, shuffle=True, **loader_kwargs)
    val = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
    test = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
    return train, val, test


def run_training(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    *,
    run_name: str,
    project: str = "CNN",
    epochs: int = 30,
    learning_rate: float = 1e-3,
    optimizer_name: str = "Adam",
    model_name: str = "BaselineCNN",
    log_every_epoch: bool = True,
    evaluate_test: bool = False,
    extra_config: dict | None = None,
) -> tuple[dict, nn.Module]:
    """Full train/val loop with Wandb logging. Returns (metrics, trained_model).

    Keep evaluate_test=False during development and HP search.
    Run evaluate_final_test() once when the model is fully finalized.
    """
    torch.manual_seed(RANDOM_SEED)

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    elif optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    batch_size = train_loader.batch_size

    wandb_config = {
        "model": model_name,
        "epochs": epochs,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "optimizer": optimizer_name,
        "loss": "CrossEntropyLoss",
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
        "test_ratio": TEST_RATIO,
        "random_seed": RANDOM_SEED,
        "device": str(device),
    }
    if extra_config:
        wandb_config.update(extra_config)

    wandb.init(
        project=project,
        name=run_name,
        config=wandb_config,
        reinit=True,
    )

    print(f"Training {run_name} for {epochs} epochs on {device}")

    best_val_acc = 0.0
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        best_val_acc = max(best_val_acc, val_acc)

        wandb.log(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_accuracy": train_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            }
        )

        if log_every_epoch:
            print(
                f"Epoch {epoch:3d}/{epochs} | "
                f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc * 100:.2f}% | "
                f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc * 100:.2f}%"
            )

    test_loss, test_acc = None, None
    if evaluate_test:
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    log_payload = {"best_val_accuracy": best_val_acc}
    if evaluate_test:
        log_payload["test_loss"] = test_loss
        log_payload["test_accuracy"] = test_acc
    wandb.log(log_payload)
    wandb.finish()

    if evaluate_test:
        print(
            f"Finished {run_name} | Best Val Acc: {best_val_acc * 100:.2f}% | "
            f"Test Acc: {test_acc * 100:.2f}%"
        )
    else:
        print(f"Finished {run_name} | Best Val Acc: {best_val_acc * 100:.2f}% (test not evaluated)")

    return {
        "run_name": run_name,
        "best_val_accuracy": best_val_acc,
        "test_accuracy": test_acc,
        "final_train_accuracy": train_acc,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "optimizer": optimizer_name,
    }, model


def evaluate_final_test(
    model: nn.Module,
    test_loader: DataLoader,
    *,
    run_name: str,
    project: str = "CNN",
    model_name: str = "BaselineCNN",
) -> tuple[float, float]:
    """Evaluate held-out test set once — only after model + hyperparameters are finalized."""
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    wandb.init(
        project=project,
        name=run_name,
        config={"model": model_name, "split": "test", "device": str(device)},
        reinit=True,
    )
    wandb.log({"test_loss": test_loss, "test_accuracy": test_acc})
    wandb.finish()

    print(f"Final test | {run_name} | Loss: {test_loss:.4f} | Acc: {test_acc * 100:.2f}%")
    return test_loss, test_acc


In [ ]:
# --- Shared experiment config (all models) ---
WANDB_PROJECT = "CNN"
EPOCHS = 30           # Models 1, 2, 4 comparison
EPOCHS_OPTIMAL = 40   # Model 3 final
EPOCHS_ATTENTION = 40 # Model 4 final

# --- Run All on a fresh Kaggle import ---
# Models 1 & 2: toggles are False → prints recorded results only (seconds).
# Model 3 test (Cell 18): needs optimal_model in memory OR set
#   RUN_OPTIMAL_FINAL_TEST=True and RUN_OPTIMAL_RETRAIN_FOR_TEST=True
#   → retrains 40 epochs then tests (~10 min). Turn both False after test is recorded.
# Model 4 (Cells 20–22): only cells with RUN_ATTENTION_* = True will train.
# Tip: set ONE active task per Run All — do not leave old True flags on.

In [ ]:
# =============================================================================
# MODEL 1 — BaselineCNN (underfitting) — COMPLETED
# =============================================================================
RUN_NAME = "Baseline_CNN"
LR = 1e-3

RUN_BASELINE_TRAINING = False
RUN_HP_VARIANTS = False
RUN_BASELINE_FINAL_TEST = False

In [ ]:
# Recorded results (from completed runs — update if you re-train)
BASELINE_VAL_ACC = 0.4624
BASELINE_TEST_ACC = 0.4541

train_loader, val_loader, test_loader = build_dataloaders(BATCH_SIZE)
baseline_model = None

if RUN_BASELINE_TRAINING:
    baseline_model = BaselineCNN()
    _, baseline_model = run_training(
        baseline_model,
        train_loader,
        val_loader,
        test_loader,
        run_name=RUN_NAME,
        project=WANDB_PROJECT,
        epochs=EPOCHS,
        learning_rate=LR,
        optimizer_name="Adam",
        model_name="BaselineCNN",
        evaluate_test=False,
    )
else:
    print(f"Model 1 done — Baseline_CNN val: {BASELINE_VAL_ACC * 100:.2f}% | test: {BASELINE_TEST_ACC * 100:.2f}%")

In [ ]:
# Cell 8 [Model 1 — optional]: LR hyperparameter comparison (already completed)
HP_RUN_NAME = "Baseline_CNN_HyperparameterSelection"

RECORDED_HP_RESULTS = [
    {"run_name": "Baseline_CNN", "learning_rate": 1e-3, "batch_size": 64, "optimizer": "Adam", "best_val_accuracy": 0.4624},
    {"run_name": f"{HP_RUN_NAME}_lr1e-2", "learning_rate": 1e-2, "batch_size": 64, "optimizer": "Adam", "best_val_accuracy": 0.3929},
    {"run_name": f"{HP_RUN_NAME}_lr5e-4", "learning_rate": 5e-4, "batch_size": 64, "optimizer": "Adam", "best_val_accuracy": 0.4389},
]

if RUN_HP_VARIANTS:
    hp_results = []
    for hp in [{"run_name": f"{HP_RUN_NAME}_lr1e-2", "learning_rate": 1e-2, "batch_size": 64, "optimizer": "Adam"},
               {"run_name": f"{HP_RUN_NAME}_lr5e-4", "learning_rate": 5e-4, "batch_size": 64, "optimizer": "Adam"}]:
        print(f"\n=== {hp['run_name']} ===")
        train_ldr, val_ldr, test_ldr = build_dataloaders(hp["batch_size"])
        metrics, _ = run_training(
            BaselineCNN(), train_ldr, val_ldr, test_ldr,
            run_name=hp["run_name"], project=WANDB_PROJECT, epochs=EPOCHS,
            learning_rate=hp["learning_rate"], optimizer_name=hp["optimizer"],
            model_name="BaselineCNN", evaluate_test=False,
        )
        hp_results.append(metrics)
    results_df = pd.DataFrame([RECORDED_HP_RESULTS[0]] + hp_results)
else:
    print("Using recorded HP results (RUN_HP_VARIANTS=False).")
    results_df = pd.DataFrame(RECORDED_HP_RESULTS)

results_df = results_df.sort_values("best_val_accuracy", ascending=False)
print("\n=== Model 1 HP summary (val accuracy) ===")
display(results_df[["run_name", "learning_rate", "batch_size", "optimizer", "best_val_accuracy"]])
best = results_df.iloc[0]
print(f"Selected config: {best['run_name']} | Val: {best['best_val_accuracy'] * 100:.2f}%")

In [ ]:
# Cell 9 [Model 1 — optional]: Final test (already completed — test acc 45.41%)
if RUN_BASELINE_FINAL_TEST:
    if baseline_model is None:
        print("Training BaselineCNN before final test...")
        baseline_model = BaselineCNN()
        _, baseline_model = run_training(
            baseline_model, train_loader, val_loader, test_loader,
            run_name=RUN_NAME, project=WANDB_PROJECT, epochs=EPOCHS,
            learning_rate=LR, optimizer_name="Adam", model_name="BaselineCNN",
            evaluate_test=False,
        )
    evaluate_final_test(
        baseline_model, test_loader,
        run_name="Baseline_CNN_FinalTest", project=WANDB_PROJECT, model_name="BaselineCNN",
    )
else:
    print(f"Baseline final test recorded: {BASELINE_TEST_ACC * 100:.2f}% (RUN_BASELINE_FINAL_TEST=False)")

In [ ]:
# =============================================================================
# MODEL 2 — Deep CNN variants (overfitting) — no BatchNorm, no Dropout
# =============================================================================
def _wide_conv_trunk() -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
        nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
        nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
        nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
        nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(inplace=True),
    )


class DeepCNN(nn.Module):
    """Wide trunk + 512-dim FC — your completed DeepCNN_NoReg run."""

    NUM_CLASSES = 7

    def __init__(self):
        super().__init__()
        self.features = _wide_conv_trunk()
        self.classifier = nn.Sequential(
            nn.Linear(512 * 3 * 3, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, self.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class DeepCNN_DeepNarrow(nn.Module):
    """6 conv layers, narrower filters — more depth, fewer params than Wide."""

    NUM_CLASSES = 7

    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 * 3 * 3, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, self.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


class DeepCNN_WideFC(nn.Module):
    """Same wide conv trunk, larger FC head — even more memorization capacity."""

    NUM_CLASSES = 7

    def __init__(self):
        super().__init__()
        self.features = _wide_conv_trunk()
        self.classifier = nn.Sequential(
            nn.Linear(512 * 3 * 3, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, self.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


DEEP_ARCHITECTURES = {
    "DeepCNN_Wide": DeepCNN,
    "DeepCNN_DeepNarrow": DeepCNN_DeepNarrow,
    "DeepCNN_WideFC": DeepCNN_WideFC,
}

for name, cls in DEEP_ARCHITECTURES.items():
    m = cls()
    print(f"{name}: {sum(p.numel() for p in m.parameters()):,} params")
    del m

In [ ]:
# Cell 12 [Model 2]: Architecture comparison — COMPLETED
# Winner by val accuracy: DeepCNN_Wide (52.58%)
DEEP_LR = 1e-3
SELECTED_DEEP_ARCH = "DeepCNN_Wide"

RUN_DEEP_ARCH_COMPARISON = False  # all 3 architectures already trained
RUN_DEEP_FINAL_TEST = False       # completed — DeepCNN_Wide_FinalTest: 50.06%

RECORDED_DEEP_RESULTS = [
    {"run_name": "DeepCNN_Wide", "best_val_accuracy": 0.5258, "final_train_accuracy": 0.9805},
    {"run_name": "DeepCNN_WideFC", "best_val_accuracy": 0.5065, "final_train_accuracy": 0.9748},
    {"run_name": "DeepCNN_DeepNarrow", "best_val_accuracy": 0.4993, "final_train_accuracy": 0.9663},
]

DEEP_VAL_ACC = 0.5258
DEEP_TEST_ACC = 0.5006  # DeepCNN_Wide_FinalTest

ARCHS_TO_TRAIN = ["DeepCNN_DeepNarrow", "DeepCNN_WideFC"]

deep_results = list(RECORDED_DEEP_RESULTS)
best_deep_model = None
best_deep_name = SELECTED_DEEP_ARCH
best_deep_val = DEEP_VAL_ACC

if RUN_DEEP_ARCH_COMPARISON:
    for arch_name in ARCHS_TO_TRAIN:
        print(f"\n=== {arch_name} ===")
        model = DEEP_ARCHITECTURES[arch_name]()
        metrics, trained_model = run_training(
            model, train_loader, val_loader, test_loader,
            run_name=arch_name, project=WANDB_PROJECT, epochs=EPOCHS,
            learning_rate=DEEP_LR, optimizer_name="Adam",
            model_name=arch_name, evaluate_test=False,
        )
        deep_results.append(
            {
                "run_name": arch_name,
                "best_val_accuracy": metrics["best_val_accuracy"],
                "final_train_accuracy": metrics["final_train_accuracy"],
            }
        )
        if metrics["best_val_accuracy"] > best_deep_val:
            best_deep_val = metrics["best_val_accuracy"]
            best_deep_model = trained_model
            best_deep_name = arch_name
else:
    print("Using recorded Model 2 results (RUN_DEEP_ARCH_COMPARISON=False).")

deep_results_df = pd.DataFrame(deep_results).sort_values("best_val_accuracy", ascending=False)
print("\n=== Model 2 architecture comparison (val accuracy) ===")
display(deep_results_df[["run_name", "best_val_accuracy", "final_train_accuracy"]])

winner = deep_results_df.iloc[0]
SELECTED_DEEP_ARCH = winner["run_name"]
print(
    f"Winner: {SELECTED_DEEP_ARCH} | Val: {winner['best_val_accuracy'] * 100:.2f}% | "
    f"Train: {winner['final_train_accuracy'] * 100:.2f}%"
)
print(f"Model 2 done — {SELECTED_DEEP_ARCH} | val: {DEEP_VAL_ACC * 100:.2f}% | test: {DEEP_TEST_ACC * 100:.2f}%")

In [ ]:
# Cell 13 [Model 2]: Final test on winner — DeepCNN_Wide — run ONCE
if RUN_DEEP_FINAL_TEST:
    if SELECTED_DEEP_ARCH not in DEEP_ARCHITECTURES:
        raise ValueError(f"Unknown architecture: {SELECTED_DEEP_ARCH}")

    print(f"Preparing {SELECTED_DEEP_ARCH} for final test (retrain 30 epochs, then evaluate test once)...")
    model = DEEP_ARCHITECTURES[SELECTED_DEEP_ARCH]()
    _, best_deep_model = run_training(
        model,
        train_loader,
        val_loader,
        test_loader,
        run_name=f"{SELECTED_DEEP_ARCH}_final_train",
        project=WANDB_PROJECT,
        epochs=EPOCHS,
        learning_rate=DEEP_LR,
        optimizer_name="Adam",
        model_name=SELECTED_DEEP_ARCH,
        evaluate_test=False,
    )

    test_loss, DEEP_TEST_ACC = evaluate_final_test(
        best_deep_model,
        test_loader,
        run_name=f"{SELECTED_DEEP_ARCH}_FinalTest",
        project=WANDB_PROJECT,
        model_name=SELECTED_DEEP_ARCH,
    )
    print(f"Model 2 complete — {SELECTED_DEEP_ARCH} | val: {DEEP_VAL_ACC * 100:.2f}% | test: {DEEP_TEST_ACC * 100:.2f}%")
else:
    print(
        f"Model 2 final test recorded: {SELECTED_DEEP_ARCH} | "
        f"val: {DEEP_VAL_ACC * 100:.2f}% | test: {DEEP_TEST_ACC * 100:.2f}% "
        f"(RUN_DEEP_FINAL_TEST=False)"
    )

In [ ]:
# =============================================================================
# MODEL 3 — OptimalCNN (best generalization) — BN + Dropout + Augmentation
# =============================================================================
def _optimal_conv_block(in_ch: int, out_ch: int, pool: bool = True, dropout: float = 0.3) -> nn.Sequential:
    layers = [
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(2, 2))
    if dropout > 0:
        layers.append(nn.Dropout2d(dropout))
    return nn.Sequential(*layers)


class OptimalCNN(nn.Module):
    """Deep CNN with BatchNorm, Dropout, and train-time augmentation (Cell 15)."""

    NUM_CLASSES = 7

    def __init__(self, dropout_fc: float = 0.4, dropout_conv: float = 0.3):
        super().__init__()
        self.features = nn.Sequential(
            _optimal_conv_block(1, 32, pool=True, dropout=dropout_conv),
            _optimal_conv_block(32, 64, pool=True, dropout=dropout_conv),
            _optimal_conv_block(64, 128, pool=True, dropout=dropout_conv),
            _optimal_conv_block(128, 256, pool=True, dropout=dropout_conv),
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_fc),
            nn.Linear(512 * 3 * 3, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_fc),
            nn.Linear(512, self.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


_optimal = OptimalCNN()
print(f"OptimalCNN output shape: {_optimal(torch.randn(2, 1, 48, 48)).shape}")
print(f"OptimalCNN parameters: {sum(p.numel() for p in _optimal.parameters()):,}")
del _optimal

In [ ]:
# Cell 15 [Model 3]: Setup — loaders, sanity check, toggles
OPTIMAL_LR = 1e-3
DROPOUT_FC = 0.4
DROPOUT_CONV = 0.3

RUN_OPTIMAL_SANITY_CHECK = False      # already passed
RUN_OPTIMAL_COMPARISON = False
RUN_OPTIMAL_FINAL_TRAINING = False    # set True to train 40 epochs (Cell 17)
RUN_OPTIMAL_FINAL_TEST = False        # set True for one-time test (Cell 18)
RUN_OPTIMAL_RETRAIN_FOR_TEST = True   # Cell 18 only: retrain if optimal_model is missing

CNN_BATCHNORM_VARIANT = "CNN_BatchNorm"
CNN_FULLLOAD_VARIANT = "CNN_FullLoad"
SELECTED_OPTIMAL_VARIANT = CNN_BATCHNORM_VARIANT  # last Model 3 run (40-epoch Optimal_CNN)
OPTIMAL_VAL_ACC = 0.5711   # CNN_BatchNorm 40 epochs — CNN_FullLoad never trained
OPTIMAL_TEST_ACC = None    # final test skipped — moving to Model 4

# Progressive configs — add regularization one step at a time (not the full recipe upfront)
OPTIMAL_VARIANTS = {
    CNN_BATCHNORM_VARIANT: {
        "dropout_fc": 0.0,
        "dropout_conv": 0.0,
        "augment": False,
        "note": "BatchNorm only — first baseline for Model 3",
    },
    "OptimalCNN_BN_Dropout": {
        "dropout_fc": 0.4,
        "dropout_conv": 0.3,
        "augment": False,
        "note": "BN + Dropout — still no augmentation",
    },
    CNN_FULLLOAD_VARIANT: {
        "dropout_fc": 0.4,
        "dropout_conv": 0.3,
        "augment": True,
        "note": "BN + Dropout + augmentation — final Model 3",
    },
}

TRAIN_AUGMENT = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=12),
    transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
])

train_loader_aug, val_loader_aug, test_loader_aug = build_augmented_dataloaders(
    BATCH_SIZE, TRAIN_AUGMENT
)


def get_optimal_loaders(use_augment: bool):
    if use_augment:
        return train_loader_aug, val_loader_aug, test_loader_aug
    return train_loader, val_loader, test_loader


optimal_model = None

if RUN_OPTIMAL_SANITY_CHECK:
    sanity_check(OptimalCNN(), train_loader, device)
else:
    print("Skipping OptimalCNN sanity check.")

In [ ]:
# Cell 16 [Model 3]: Regularization comparison — progressive stages (30 epochs, val only)
# Train one variant at a time: BN only → + Dropout → + Augmentation. Pick winner by val accuracy.

VARIANTS_TO_TRAIN = []  # comparison done for now — full config trains in Cell 17

# Completed comparison runs (CNN_FullLoad not run — see Model 4 for BN+dropout+aug)
RECORDED_OPTIMAL_RESULTS = [
    {
        "run_name": "CNN_BatchNorm",
        "dropout_fc": 0.0,
        "dropout_conv": 0.0,
        "augmentation": "none",
        "best_val_accuracy": 0.5769,
        "final_train_accuracy": 0.9778,
        "note": "30-epoch comparison",
    },
    {
        "run_name": "CNN_BatchNorm",
        "dropout_fc": 0.0,
        "dropout_conv": 0.0,
        "augmentation": "none",
        "best_val_accuracy": 0.5711,
        "final_train_accuracy": 0.9889,
        "note": "40-epoch Optimal_CNN (Cell 17 bug used BatchNorm config)",
    },
]

optimal_results = list(RECORDED_OPTIMAL_RESULTS)
best_optimal_model = None
best_optimal_name = SELECTED_OPTIMAL_VARIANT
best_optimal_val = max((r["best_val_accuracy"] for r in optimal_results), default=0.0)

if RUN_OPTIMAL_COMPARISON:
    for variant_name in VARIANTS_TO_TRAIN:
        if variant_name not in OPTIMAL_VARIANTS:
            raise ValueError(f"Unknown variant: {variant_name}")

        cfg = OPTIMAL_VARIANTS[variant_name]
        train_ldr, val_ldr, test_ldr = get_optimal_loaders(cfg["augment"])
        aug_label = "flip+rotation+affine" if cfg["augment"] else "none"

        print(f"\n=== {variant_name} | {cfg['note']} ===")
        model = OptimalCNN(dropout_fc=cfg["dropout_fc"], dropout_conv=cfg["dropout_conv"])
        metrics, trained_model = run_training(
            model,
            train_ldr,
            val_ldr,
            test_ldr,
            run_name=variant_name,
            project=WANDB_PROJECT,
            epochs=EPOCHS,
            learning_rate=OPTIMAL_LR,
            optimizer_name="Adam",
            model_name="OptimalCNN",
            evaluate_test=False,
            extra_config={
                "variant": variant_name,
                "dropout_fc": cfg["dropout_fc"],
                "dropout_conv": cfg["dropout_conv"],
                "augmentation": aug_label,
                "comparison_phase": True,
            },
        )
        optimal_results.append(
            {
                "run_name": variant_name,
                "dropout_fc": cfg["dropout_fc"],
                "dropout_conv": cfg["dropout_conv"],
                "augmentation": aug_label,
                "best_val_accuracy": metrics["best_val_accuracy"],
                "final_train_accuracy": metrics["final_train_accuracy"],
            }
        )
        if metrics["best_val_accuracy"] > best_optimal_val:
            best_optimal_val = metrics["best_val_accuracy"]
            best_optimal_model = trained_model
            best_optimal_name = variant_name
else:
    print("Skipping Model 3 comparison (RUN_OPTIMAL_COMPARISON=False).")

if optimal_results:
    optimal_results_df = pd.DataFrame(optimal_results).sort_values(
        "best_val_accuracy", ascending=False
    )
    print("\n=== Model 3 regularization comparison (val accuracy) ===")
    display(
        optimal_results_df[
            [
                "run_name",
                "dropout_fc",
                "dropout_conv",
                "augmentation",
                "best_val_accuracy",
                "final_train_accuracy",
            ]
        ]
    )

    winner = optimal_results_df.iloc[0]
    if RUN_OPTIMAL_COMPARISON:
        SELECTED_OPTIMAL_VARIANT = winner["run_name"]
        OPTIMAL_VAL_ACC = winner["best_val_accuracy"]
    print(
        f"Comparison leader: {winner['run_name']} | "
        f"Val: {winner['best_val_accuracy'] * 100:.2f}% | "
        f"Train: {winner['final_train_accuracy'] * 100:.2f}%"
    )
    if not RUN_OPTIMAL_COMPARISON:
        print(f"Final training variant (Cell 15): {SELECTED_OPTIMAL_VARIANT}")
else:
    print("No Model 3 comparison runs yet.")

In [ ]:
# Cell 17 [Model 3]: Final training — winner config, 40 epochs (after comparison is done)
OPTIMAL_RUN_NAME = "Optimal_CNN"

if RUN_OPTIMAL_FINAL_TRAINING:
    if SELECTED_OPTIMAL_VARIANT not in OPTIMAL_VARIANTS:
        raise ValueError(f"Unknown variant: {SELECTED_OPTIMAL_VARIANT}")

    cfg = OPTIMAL_VARIANTS[SELECTED_OPTIMAL_VARIANT]
    train_ldr, val_ldr, test_ldr = get_optimal_loaders(cfg["augment"])
    aug_label = "flip+rotation+affine" if cfg["augment"] else "none"

    print(
        f"Final Model 3 training: {SELECTED_OPTIMAL_VARIANT} | "
        f"{cfg['note']} | {EPOCHS_OPTIMAL} epochs"
    )
    optimal_model = OptimalCNN(
        dropout_fc=cfg["dropout_fc"], dropout_conv=cfg["dropout_conv"]
    )
    optimal_metrics, optimal_model = run_training(
        optimal_model,
        train_ldr,
        val_ldr,
        test_ldr,
        run_name=OPTIMAL_RUN_NAME,
        project=WANDB_PROJECT,
        epochs=EPOCHS_OPTIMAL,
        learning_rate=OPTIMAL_LR,
        optimizer_name="Adam",
        model_name="OptimalCNN",
        evaluate_test=False,
        extra_config={
            "variant": SELECTED_OPTIMAL_VARIANT,
            "dropout_fc": cfg["dropout_fc"],
            "dropout_conv": cfg["dropout_conv"],
            "augmentation": aug_label,
            "comparison_phase": False,
        },
    )
    OPTIMAL_VAL_ACC = optimal_metrics["best_val_accuracy"]
    print(
        f"OptimalCNN final training done — best val: {OPTIMAL_VAL_ACC * 100:.2f}% | "
        f"Goal: beat Model 2 val (~52.6%) with a smaller train–val gap."
    )
else:
    print("Skipping OptimalCNN final training (RUN_OPTIMAL_FINAL_TRAINING=False).")

In [ ]:
# Cell 18 [Model 3]: Final test — run ONCE after 40-epoch training is complete
if RUN_OPTIMAL_FINAL_TEST:
    cfg = OPTIMAL_VARIANTS[SELECTED_OPTIMAL_VARIANT]
    _, _, test_ldr = get_optimal_loaders(cfg["augment"])

    if optimal_model is None:
        if not RUN_OPTIMAL_RETRAIN_FOR_TEST:
            print(
                "Model 3 final test skipped — optimal_model not in memory. "
                "On a fresh Kaggle import set RUN_OPTIMAL_RETRAIN_FOR_TEST=True "
                "(retrains ~10 min) or run Cell 17 with RUN_OPTIMAL_FINAL_TRAINING=True first."
            )
        else:
            print("Fresh session: training OptimalCNN before final test (~40 epochs)...")
            train_ldr, val_ldr, _ = get_optimal_loaders(cfg["augment"])
            aug_label = "flip+rotation+affine" if cfg["augment"] else "none"
            optimal_model = OptimalCNN(
                dropout_fc=cfg["dropout_fc"], dropout_conv=cfg["dropout_conv"]
            )
            _, optimal_model = run_training(
                optimal_model,
                train_ldr,
                val_ldr,
                test_ldr,
                run_name=f"{OPTIMAL_RUN_NAME}_final_train",
                project=WANDB_PROJECT,
                epochs=EPOCHS_OPTIMAL,
                learning_rate=OPTIMAL_LR,
                optimizer_name="Adam",
                model_name="OptimalCNN",
                evaluate_test=False,
                extra_config={
                    "variant": SELECTED_OPTIMAL_VARIANT,
                    "dropout_fc": cfg["dropout_fc"],
                    "dropout_conv": cfg["dropout_conv"],
                    "augmentation": aug_label,
                },
            )

    if optimal_model is not None:
        test_loss, OPTIMAL_TEST_ACC = evaluate_final_test(
            optimal_model,
            test_ldr,
            run_name="Optimal_CNN_FinalTest",
            project=WANDB_PROJECT,
            model_name="OptimalCNN",
        )
        print(
            f"Model 3 complete — {SELECTED_OPTIMAL_VARIANT} | "
            f"val: {OPTIMAL_VAL_ACC * 100:.2f}% | test: {OPTIMAL_TEST_ACC * 100:.2f}%"
        )
else:
    if OPTIMAL_TEST_ACC is not None:
        print(
            f"Model 3 final test recorded: {SELECTED_OPTIMAL_VARIANT} | "
            f"val: {OPTIMAL_VAL_ACC * 100:.2f}% | test: {OPTIMAL_TEST_ACC * 100:.2f}% "
            f"(RUN_OPTIMAL_FINAL_TEST=False)"
        )
    else:
        print("OptimalCNN final test not run (RUN_OPTIMAL_FINAL_TEST=False).")

In [ ]:
# =============================================================================
# MODEL 4 — CNN + Attention (plug-and-play on OptimalCNN blocks)
# SE-Net: channel attention | CBAM: channel + spatial | ECA-Net: lightweight channel
# =============================================================================
class SEBlock(nn.Module):
    """Squeeze-and-Excitation — channel-wise recalibration."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1)
        return x * w


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.mlp = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        avg = self.mlp(self.avg_pool(x).view(b, c))
        mx = self.mlp(self.max_pool(x).view(b, c))
        return torch.sigmoid(avg + mx).view(b, c, 1, 1)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        attn = self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))
        return attn


class CBAM(nn.Module):
    """Convolutional Block Attention Module — channel then spatial."""

    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.channel = ChannelAttention(channels, reduction)
        self.spatial = SpatialAttention()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x * self.channel(x)
        x = x * self.spatial(x)
        return x


class ECABlock(nn.Module):
    """Efficient Channel Attention — 1D conv on channel descriptor, minimal params."""

    def __init__(self, channels: int, k_size: int = 3):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(
            1, 1, kernel_size=k_size, padding=k_size // 2, bias=False
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        y = self.pool(x).view(b, 1, c)
        y = self.sigmoid(self.conv(y)).view(b, c, 1, 1)
        return x * y


def _make_attention_block(attention_type: str, channels: int) -> nn.Module:
    if attention_type == "se":
        return SEBlock(channels)
    if attention_type == "cbam":
        return CBAM(channels)
    if attention_type == "eca":
        return ECABlock(channels)
    raise ValueError(f"Unknown attention_type: {attention_type}")


class AttentionOptimalCNN(nn.Module):
    """OptimalCNN trunk with SE / CBAM / ECA after each conv stage."""

    NUM_CLASSES = 7
    STAGES = [(1, 32), (32, 64), (64, 128), (128, 256)]

    def __init__(
        self,
        attention_type: str = "se",
        dropout_fc: float = 0.4,
        dropout_conv: float = 0.3,
    ):
        super().__init__()
        self.attention_type = attention_type
        self.blocks = nn.ModuleList()
        self.attentions = nn.ModuleList()
        for in_ch, out_ch in self.STAGES:
            self.blocks.append(
                _optimal_conv_block(in_ch, out_ch, pool=True, dropout=dropout_conv)
            )
            self.attentions.append(_make_attention_block(attention_type, out_ch))
        self.head = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            SEBlock(512),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_fc),
            nn.Linear(512 * 3 * 3, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_fc),
            nn.Linear(512, self.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block, attn in zip(self.blocks, self.attentions):
            x = attn(block(x))
        x = self.head(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


ATTENTION_ARCHITECTURES = {
    "CNN_Attention_SE": lambda: AttentionOptimalCNN("se"),
    "CNN_Attention_CBAM": lambda: AttentionOptimalCNN("cbam"),
    "CNN_Attention_ECA": lambda: AttentionOptimalCNN("eca"),
}

for name, factory in ATTENTION_ARCHITECTURES.items():
    m = factory()
    print(f"{name}: {sum(p.numel() for p in m.parameters()):,} params")
    del m

In [ ]:
# Cell 20 [Model 4]: Setup — same regularization as CNN_FullLoad (BN+dropout+aug)
# CNN_FullLoad was never run standalone; Model 4 trains that recipe + attention.
ATTENTION_LR = 1e-3
ATTENTION_DROPOUT_FC = 0.4
ATTENTION_DROPOUT_CONV = 0.3

RUN_ATTENTION_SANITY_CHECK = False     # passed
RUN_ATTENTION_COMPARISON = False       # all 3 architectures done — SE wins

SELECTED_ATTENTION_ARCH = "CNN_Attention_SE"  # winner: 59.64% val
ATTENTION_VAL_ACC = 0.5964
ATTENTION_TEST_ACC = None

CKPT_DIR = Path("/kaggle/working/checkpoints") if Path("/kaggle/working").exists() else Path("checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
ATTENTION_CKPT = CKPT_DIR / "cnn_attention_se_final.pt"

In [ ]:
# Cell 21 [Model 4]: Attention comparison — train one architecture at a time
ATTENTION_ARCHS_TO_TRAIN = []

RECORDED_ATTENTION_RESULTS = [
    {
        "run_name": "CNN_Attention_SE",
        "best_val_accuracy": 0.5964,
        "final_train_accuracy": 0.5510,
    },
    {
        "run_name": "CNN_Attention_CBAM",
        "best_val_accuracy": 0.5954,
        "final_train_accuracy": 0.5469,
    },
    {
        "run_name": "CNN_Attention_ECA",
        "best_val_accuracy": 0.5936,
        "final_train_accuracy": 0.5522,
    },
]

attention_results = list(RECORDED_ATTENTION_RESULTS)
best_attention_model = None
best_attention_name = SELECTED_ATTENTION_ARCH
best_attention_val = max((r["best_val_accuracy"] for r in attention_results), default=0.0)

if RUN_ATTENTION_SANITY_CHECK:
    sanity_check(AttentionOptimalCNN("se"), train_loader_aug, device)
else:
    print("Skipping attention sanity check.")

if RUN_ATTENTION_COMPARISON:
    for arch_name in ATTENTION_ARCHS_TO_TRAIN:
        if arch_name not in ATTENTION_ARCHITECTURES:
            raise ValueError(f"Unknown architecture: {arch_name}")

        print(f"\n=== {arch_name} ===")
        model = ATTENTION_ARCHITECTURES[arch_name]()
        metrics, trained_model = run_training(
            model,
            train_loader_aug,
            val_loader_aug,
            test_loader_aug,
            run_name=arch_name,
            project=WANDB_PROJECT,
            epochs=EPOCHS,
            learning_rate=ATTENTION_LR,
            optimizer_name="Adam",
            model_name=arch_name,
            evaluate_test=False,
            extra_config={
                "attention": arch_name.replace("CNN_Attention_", "").lower(),
                "dropout_fc": ATTENTION_DROPOUT_FC,
                "dropout_conv": ATTENTION_DROPOUT_CONV,
                "augmentation": "flip+rotation+affine",
                "comparison_phase": True,
            },
        )
        attention_results.append(
            {
                "run_name": arch_name,
                "best_val_accuracy": metrics["best_val_accuracy"],
                "final_train_accuracy": metrics["final_train_accuracy"],
            }
        )
        if metrics["best_val_accuracy"] > best_attention_val:
            best_attention_val = metrics["best_val_accuracy"]
            best_attention_model = trained_model
            best_attention_name = arch_name
else:
    print("Skipping Model 4 comparison (RUN_ATTENTION_COMPARISON=False).")

if attention_results:
    attention_results_df = pd.DataFrame(attention_results).sort_values(
        "best_val_accuracy", ascending=False
    )
    print("\n=== Model 4 attention comparison (val accuracy) ===")
    display(
        attention_results_df[["run_name", "best_val_accuracy", "final_train_accuracy"]]
    )
    winner = attention_results_df.iloc[0]
    if RUN_ATTENTION_COMPARISON:
        SELECTED_ATTENTION_ARCH = winner["run_name"]
        ATTENTION_VAL_ACC = winner["best_val_accuracy"]
    print(
        f"Leader: {winner['run_name']} | Val: {winner['best_val_accuracy'] * 100:.2f}% | "
        f"Train: {winner['final_train_accuracy'] * 100:.2f}%"
    )
    if not RUN_ATTENTION_COMPARISON:
        print(f"Final training arch (Cell 20): {SELECTED_ATTENTION_ARCH}")
else:
    print("No Model 4 comparison runs yet.")

In [ ]:
# Cell 22 [Model 4]: Final training + test — ALL toggles live here (do not split across Cell 20)
RUN_ATTENTION_FINAL_TRAINING = True   # 40-epoch train
RUN_ATTENTION_FINAL_TEST = True        # test immediately after train (same cell run)
RUN_ATTENTION_RETRAIN_FOR_TEST = False  # if True: retrain when no model/checkpoint (slow)

ATTENTION_RUN_NAME = "CNN_Attention_Best"


def _save_attention_ckpt(model: nn.Module) -> None:
    torch.save(
        {"arch": SELECTED_ATTENTION_ARCH, "state_dict": model.state_dict()},
        ATTENTION_CKPT,
    )
    print(f"Saved checkpoint → {ATTENTION_CKPT}")


def _load_attention_ckpt() -> nn.Module | None:
    if not ATTENTION_CKPT.exists():
        return None
    payload = torch.load(ATTENTION_CKPT, map_location=device)
    arch = payload.get("arch", SELECTED_ATTENTION_ARCH)
    model = ATTENTION_ARCHITECTURES[arch]()
    model.load_state_dict(payload["state_dict"])
    model = model.to(device)
    print(f"Loaded checkpoint ← {ATTENTION_CKPT}")
    return model


if RUN_ATTENTION_FINAL_TRAINING:
    print(f"Final Model 4 training: {SELECTED_ATTENTION_ARCH} | {EPOCHS_ATTENTION} epochs")
    attention_model = ATTENTION_ARCHITECTURES[SELECTED_ATTENTION_ARCH]()
    attn_metrics, attention_model = run_training(
        attention_model,
        train_loader_aug,
        val_loader_aug,
        test_loader_aug,
        run_name=ATTENTION_RUN_NAME,
        project=WANDB_PROJECT,
        epochs=EPOCHS_ATTENTION,
        learning_rate=ATTENTION_LR,
        optimizer_name="Adam",
        model_name=SELECTED_ATTENTION_ARCH,
        evaluate_test=False,
        extra_config={
            "attention": SELECTED_ATTENTION_ARCH.replace("CNN_Attention_", "").lower(),
            "dropout_fc": ATTENTION_DROPOUT_FC,
            "dropout_conv": ATTENTION_DROPOUT_CONV,
            "augmentation": "flip+rotation+affine",
            "comparison_phase": False,
        },
    )
    ATTENTION_VAL_ACC = attn_metrics["best_val_accuracy"]
    _save_attention_ckpt(attention_model)
    print(f"Model 4 final training done — best val: {ATTENTION_VAL_ACC * 100:.2f}%")
else:
    print("Skipping Model 4 final training (RUN_ATTENTION_FINAL_TRAINING=False).")

if RUN_ATTENTION_FINAL_TEST:
    if "attention_model" not in globals() or attention_model is None:
        attention_model = _load_attention_ckpt()
    if attention_model is None and RUN_ATTENTION_RETRAIN_FOR_TEST:
        print("No model/checkpoint — training 40 epochs before test...")
        attention_model = ATTENTION_ARCHITECTURES[SELECTED_ATTENTION_ARCH]()
        attn_metrics, attention_model = run_training(
            attention_model,
            train_loader_aug,
            val_loader_aug,
            test_loader_aug,
            run_name=ATTENTION_RUN_NAME,
            project=WANDB_PROJECT,
            epochs=EPOCHS_ATTENTION,
            learning_rate=ATTENTION_LR,
            optimizer_name="Adam",
            model_name=SELECTED_ATTENTION_ARCH,
            evaluate_test=False,
            extra_config={
                "attention": SELECTED_ATTENTION_ARCH.replace("CNN_Attention_", "").lower(),
                "dropout_fc": ATTENTION_DROPOUT_FC,
                "dropout_conv": ATTENTION_DROPOUT_CONV,
                "augmentation": "flip+rotation+affine",
            },
        )
        ATTENTION_VAL_ACC = attn_metrics["best_val_accuracy"]
        _save_attention_ckpt(attention_model)
    if attention_model is None:
        raise RuntimeError(
            "No trained model. Set RUN_ATTENTION_FINAL_TRAINING=True in this cell, "
            "or RUN_ATTENTION_RETRAIN_FOR_TEST=True, or ensure checkpoint exists."
        )

    test_loss, ATTENTION_TEST_ACC = evaluate_final_test(
        attention_model,
        test_loader_aug,
        run_name=f"{SELECTED_ATTENTION_ARCH}_FinalTest",
        project=WANDB_PROJECT,
        model_name=SELECTED_ATTENTION_ARCH,
    )
    print(
        f"Model 4 complete — {SELECTED_ATTENTION_ARCH} | "
        f"val: {ATTENTION_VAL_ACC * 100:.2f}% | test: {ATTENTION_TEST_ACC * 100:.2f}%"
    )
elif ATTENTION_TEST_ACC is not None:
    print(
        f"Model 4 final test recorded: {SELECTED_ATTENTION_ARCH} | "
        f"val: {ATTENTION_VAL_ACC * 100:.2f}% | test: {ATTENTION_TEST_ACC * 100:.2f}% "
        f"(RUN_ATTENTION_FINAL_TEST=False)"
    )
else:
    print("Model 4 final test not run (RUN_ATTENTION_FINAL_TEST=False).")

# MODEL 5 — CNN + Transformer hybrids (planned)

**Goal:** combine local CNN features with global Transformer context — strong on FER benchmarks when data and compute allow.

| Architecture | Idea | Notes for this project |
|---|---|---|
| **POSTER / POSTER++** | Pyramid fusion of landmark + image features via Transformer | SOTA on RAF-DB/FER2013 (~92%); needs landmarks or a strong backbone — heavier than Models 1–4 |
| **EfficientFace** | Lightweight CNN + cross-spatial attention | Good middle ground before full POSTER++ |
| **Former-DFER** | Video / dynamic FER | Skip unless we add temporal data |

**Suggested order after Model 4:** (1) EfficientFace-style hybrid as a lighter step, (2) POSTER++ if Kaggle GPU time allows.

Cells not implemented yet — finish Model 4 comparison + final test first.